# 06 - Exportar a TFLite

Convierte ambos modelos a `.tflite` con cuantizacion int8 dinamica. Verifica tamaño y prueba inferencia.


In [ ]:
!pip install -q tensorflow

In [ ]:
import tensorflow as tf
import numpy as np
from pathlib import Path

OUT = Path("./outputs")

tf.keras.mixed_precision.set_global_policy("float32")


class AsymmetricLoss(tf.keras.losses.Loss):
    def __init__(self, gamma_neg=4.0, gamma_pos=0.0, clip=0.05, label_smoothing=0.05,
                 class_weights=None, name="asymmetric_loss", **kwargs):
        super().__init__(name=name, **kwargs)
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip = clip
        self.label_smoothing = label_smoothing
        self.class_weights = class_weights
        self.eps = 1e-8

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.cast(y_pred, tf.float32)
        num_classes = tf.cast(tf.shape(y_true)[-1], tf.float32)
        y_true_s = y_true * (1.0 - self.label_smoothing) + self.label_smoothing / num_classes
        xs_pos = y_pred
        xs_neg = 1.0 - y_pred
        if self.clip > 0:
            xs_neg = tf.clip_by_value(xs_neg + self.clip, 0.0, 1.0)
        log_pos = tf.math.log(tf.clip_by_value(xs_pos, self.eps, 1.0))
        log_neg = tf.math.log(tf.clip_by_value(xs_neg, self.eps, 1.0))
        loss_pos = y_true_s * log_pos
        loss_neg = (1.0 - y_true_s) * log_neg
        if self.gamma_pos > 0:
            loss_pos = loss_pos * tf.pow(1.0 - xs_pos, self.gamma_pos)
        if self.gamma_neg > 0:
            loss_neg = loss_neg * tf.pow(1.0 - xs_neg, self.gamma_neg)
        per_class = loss_pos + loss_neg
        if self.class_weights is not None:
            per_class = per_class * tf.constant(self.class_weights, tf.float32)
        return -tf.reduce_sum(per_class, axis=-1)

    def get_config(self):
        base = super().get_config()
        base.update({
            "gamma_neg": self.gamma_neg,
            "gamma_pos": self.gamma_pos,
            "clip": self.clip,
            "label_smoothing": self.label_smoothing,
            "class_weights": self.class_weights,
        })
        return base


_custom = {"AsymmetricLoss": AsymmetricLoss}

m1_orig = tf.keras.models.load_model(OUT / "model1_binary.keras")
m2_orig = tf.keras.models.load_model(OUT / "model2_pathogen.keras", custom_objects=_custom)
print("Modelos cargados:", len(m1_orig.layers), "layers M1,", len(m2_orig.layers), "layers M2")


In [ ]:
def reconstruir_float32(original, num_clases, model_name, backbone="b0", img_size=(224, 224), multilabel=False):
    if backbone == "b1":
        base = tf.keras.applications.EfficientNetB1(
            weights=None, include_top=False, input_shape=(*img_size, 3)
        )
    else:
        base = tf.keras.applications.EfficientNetB0(
            weights=None, include_top=False, input_shape=(*img_size, 3)
        )
    x = base.output
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.4)(x)
    x = tf.keras.layers.Dense(
        256, activation="relu",
        kernel_regularizer=tf.keras.regularizers.l2(1e-4)
    )(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    if num_clases == 1 or multilabel:
        out = tf.keras.layers.Dense(num_clases, activation="sigmoid")(x)
    else:
        out = tf.keras.layers.Dense(num_clases, activation="softmax")(x)
    nuevo = tf.keras.Model(base.input, out, name=model_name)
    nuevo.set_weights(original.get_weights())
    activation = "sigmoid" if multilabel or num_clases == 1 else "softmax"
    print(f"  {model_name}: pesos copiados OK ({backbone.upper()} {img_size}, {activation})")
    return nuevo


m1 = reconstruir_float32(
    m1_orig, num_clases=1, model_name="model1_binary",
    backbone="b1", img_size=(240, 240),
)
m2 = reconstruir_float32(
    m2_orig, num_clases=5, model_name="model2_pathogen",
    backbone="b0", img_size=(224, 224), multilabel=True,
)

In [ ]:
def export_tflite(model, path, target_mb=None):
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    print(f"  Convirtiendo {path.name}...")
    tflite_model = converter.convert()
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_bytes(tflite_model)
    size_mb = Path(path).stat().st_size / (1024 * 1024)
    status = f" [{'OK' if size_mb < target_mb else 'GRANDE'}]" if target_mb else ""
    print(f"    {path.name}: {size_mb:.2f} MB{status}")
    return size_mb


export_tflite(m1, OUT / "model1.tflite", target_mb=5)
export_tflite(m2, OUT / "model2.tflite", target_mb=10)
print("\nExport completado. Copiar a App/assets/models/")


In [ ]:
from tensorflow.keras.applications.efficientnet import preprocess_input as efn_preprocess
from PIL import Image as PILImage


def representative_dataset_factory(image_dir: Path, target_size=(224, 224), max_samples: int = 100):
    paths = []
    for cls_dir in sorted(p for p in image_dir.iterdir() if p.is_dir()):
        for ext in ("*.jpg", "*.jpeg", "*.png", "*.bmp"):
            paths.extend(cls_dir.glob(ext))
    paths = paths[:max_samples]

    def generator():
        for path in paths:
            img = np.array(PILImage.open(path).convert("RGB").resize(target_size))
            arr = efn_preprocess(img.astype(np.float32))
            yield [np.expand_dims(arr, 0)]

    return generator


def export_tflite_int8(model, path, representative_data_fn):
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = representative_data_fn
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.uint8
    converter.inference_output_type = tf.uint8
    tflite_model = converter.convert()
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_bytes(tflite_model)
    size_mb = Path(path).stat().st_size / (1024 * 1024)
    print(f"  {path.name}: {size_mb:.2f} MB (int8)")
    return size_mb


VAL_DIR_M2 = Path("./splits/clasificacion_patogeno/val")
if VAL_DIR_M2.exists():
    rep_fn_m2 = representative_dataset_factory(VAL_DIR_M2, target_size=(224, 224))
    export_tflite_int8(m2, OUT / "model2_int8.tflite", rep_fn_m2)
else:
    print("WARNING: val dataset M2 no encontrado. Saltando int8 M2.")

VAL_DIR_M1 = Path("./splits/clasificacion_binaria/val")
if VAL_DIR_M1.exists():
    rep_fn_m1 = representative_dataset_factory(VAL_DIR_M1, target_size=(240, 240))
    export_tflite_int8(m1, OUT / "model1_int8.tflite", rep_fn_m1)
else:
    print("WARNING: val dataset M1 no encontrado. Saltando int8 M1.")

In [ ]:
def labels_from_indices(path_in, path_out):
    import json
    with open(path_in) as f:
        idx = json.load(f)
    sorted_labels = [k for k, _ in sorted(idx.items(), key=lambda kv: kv[1])]
    Path(path_out).write_text("\n".join(f"{i} {lbl}" for i, lbl in enumerate(sorted_labels)))
    print(f"  {path_out}: {sorted_labels}")


labels_from_indices(OUT / "class_indices_model1_binary.json", OUT / "labels_m1.txt")
labels_from_indices(OUT / "class_indices_model2_pathogen.json", OUT / "labels_m2.txt")


In [ ]:
import shutil

thresholds_src = OUT / "thresholds.json"
if thresholds_src.exists():
    print(f"thresholds.json detectado: {thresholds_src}")
    print("Copiar a App/assets/models/pd/thresholds.json junto con model_unquant.tflite y labels.txt")
else:
    print("WARNING: thresholds.json no encontrado. Ejecuta primero la calibración en 03_train_model2_pathogen.ipynb")

In [ ]:
dummy_m1 = np.random.rand(1, 240, 240, 3).astype(np.float32)
dummy_m2 = np.random.rand(1, 224, 224, 3).astype(np.float32)

inter1 = tf.lite.Interpreter(model_path=str(OUT / "model1.tflite"))
inter1.allocate_tensors()
inp1 = inter1.get_input_details()[0]
out1 = inter1.get_output_details()[0]
print("M1 Input:", inp1["shape"], inp1["dtype"])
print("M1 Output:", out1["shape"], out1["dtype"])
inter1.set_tensor(inp1["index"], dummy_m1)
inter1.invoke()
print("M1 salida prueba:", inter1.get_tensor(out1["index"]))

inter2 = tf.lite.Interpreter(model_path=str(OUT / "model2.tflite"))
inter2.allocate_tensors()
inp2 = inter2.get_input_details()[0]
out2 = inter2.get_output_details()[0]
print("M2 Input:", inp2["shape"], inp2["dtype"])
print("M2 Output:", out2["shape"], out2["dtype"])
inter2.set_tensor(inp2["index"], dummy_m2)
inter2.invoke()
print("M2 salida prueba:", inter2.get_tensor(out2["index"]))

## Copiar a App/

```
App/assets/models/hs/model.tflite          <- outputs/model1.tflite
App/assets/models/hs/labels.txt            <- outputs/labels_m1.txt
App/assets/models/pd/model_unquant.tflite  <- outputs/model2.tflite
App/assets/models/pd/labels.txt            <- outputs/labels_m2.txt
```


In [ ]:
import tensorflow as tf
import numpy as np
from pathlib import Path
from PIL import Image

OUT = globals().get('OUT', Path('./outputs'))
SPLIT = Path('./splits')

_SEG_MODEL_PATH = OUT / 'model_seg.keras'
if not _SEG_MODEL_PATH.exists():
    _SEG_MODEL_PATH = OUT / 'model_seg_best.keras'

if not _SEG_MODEL_PATH.exists():
    print('WARNING: model_seg.keras no encontrado. Ejecutar notebook 07 primero.')
else:
    def dice_loss_stub(y_true, y_pred):
        return tf.constant(0.0)

    def seg_loss_stub(y_true, y_pred):
        return tf.constant(0.0)

    class _DiceCoef(tf.keras.metrics.Metric):
        def __init__(self, class_id=0, name=None, **kw):
            super().__init__(name=name or f'dice_c{class_id}', **kw)
            self.class_id = class_id
            self._v = self.add_weight(name='v', shape=(), initializer='zeros')

        def get_config(self):
            base = super().get_config()
            base['class_id'] = self.class_id
            return base

        def update_state(self, *a, **kw): pass
        def result(self): return self._v
        def reset_state(self): self._v.assign(0.0)

    _custom_seg = {
        'seg_loss': seg_loss_stub, 'dice_loss': dice_loss_stub,
        'DiceCoef': _DiceCoef,
    }
    mseg_export = tf.keras.models.load_model(_SEG_MODEL_PATH, custom_objects=_custom_seg)
    print(f'M_seg cargado ({len(mseg_export.layers)} layers)')

    converter_f32 = tf.lite.TFLiteConverter.from_keras_model(mseg_export)
    tflite_f32 = converter_f32.convert()
    (OUT / 'model_seg.tflite').write_bytes(tflite_f32)
    size_f32 = (OUT / 'model_seg.tflite').stat().st_size / (1024 * 1024)
    print(f'model_seg.tflite (float32): {size_f32:.2f} MB')

    _val_dir = SPLIT / 'clasificacion_patogeno' / 'val'
    _val_images = []
    if _val_dir.exists():
        for cls_dir in _val_dir.iterdir():
            if cls_dir.is_dir():
                for ext in ('*.jpg', '*.jpeg', '*.png'):
                    _val_images.extend(list(cls_dir.glob(ext))[:10])
    _val_images = _val_images[:100]

    if _val_images:
        def _rep_ds_seg():
            for fp in _val_images:
                img = np.array(Image.open(fp).convert('RGB').resize((256, 256)))
                yield [img.astype(np.float32)[np.newaxis] / 255.0]

        converter_int8 = tf.lite.TFLiteConverter.from_keras_model(mseg_export)
        converter_int8.optimizations = [tf.lite.Optimize.DEFAULT]
        converter_int8.representative_dataset = _rep_ds_seg
        converter_int8.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        converter_int8.inference_input_type  = tf.uint8
        converter_int8.inference_output_type = tf.uint8
        tflite_int8 = converter_int8.convert()
        (OUT / 'model_seg_int8.tflite').write_bytes(tflite_int8)
        size_int8 = (OUT / 'model_seg_int8.tflite').stat().st_size / (1024 * 1024)
        print(f'model_seg_int8.tflite (int8):   {size_int8:.2f} MB')
    else:
        print('WARNING: imagenes de val no encontradas. Saltando int8 M_seg.')

    print('\nCopiar a App/assets/models/seg/model_seg.tflite  (preferir int8 si disponible)')